In [3]:
!pip install datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")

print(dataset)

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [6]:
print(dataset["train"][0])

{'text': 'i didnt feel humiliated', 'label': 0}


In [7]:
print(dataset["train"].features["label"].names)

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [8]:
import pandas as pd

df = pd.DataFrame(dataset["train"])

df["emotion"] = df["label"].map(
    dict(enumerate(dataset["train"].features["label"].names))
)

df.head()

,text,label,emotion
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned...,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i am ever feeling nostalgic about the fireplac...,2,love
4,i am feeling grouchy,3,anger


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     16000 non-null  object
 1   label    16000 non-null  int64 
 2   emotion  16000 non-null  object
dtypes: int64(1), object(2)
memory usage: 375.1+ KB


In [10]:
df["emotion"].value_counts()

,count
emotion,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


In [13]:
!pip install nltk

In [16]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [17]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess_text)

df[["text", "clean_text"]].head()

,text,clean_text
0,i didnt feel humiliated,didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,go feeling hopeless damned hopeful around some...
2,im grabbing a minute to post i feel greedy wrong,im grabbing minute post feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,ever feeling nostalgic fireplace know still pr...
4,i am feeling grouchy,feeling grouchy


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.15, random_state=42, stratify=df['label']
)

# TF-IDF
tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)
print("=== Naive Bayes ===")
print(f"Accuracy: {accuracy_score(y_test, nb_pred):.4f}")

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)
print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")

# SVM
svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)
svm_pred = svm.predict(X_test_tfidf)
print("=== SVM ===")
print(f"Accuracy: {accuracy_score(y_test, svm_pred):.4f}")

=== Naive Bayes ===
Accuracy: 0.6871
=== Logistic Regression ===
Accuracy: 0.8712
=== SVM ===
Accuracy: 0.8946


In [19]:
from sklearn.metrics import classification_report

print("=== Logistic Regression Report ===")
print(classification_report(y_test, lr_pred))

print("=== SVM Report ===")
print(classification_report(y_test, svm_pred))

=== Logistic Regression Report ===
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       700
           1       0.84      0.95      0.89       804
           2       0.83      0.63      0.72       196
           3       0.90      0.85      0.88       324
           4       0.87      0.78      0.82       290
           5       0.89      0.55      0.68        86

    accuracy                           0.87      2400
   macro avg       0.87      0.78      0.82      2400
weighted avg       0.87      0.87      0.87      2400

=== SVM Report ===
              precision    recall  f1-score   support

           0       0.94      0.93      0.94       700
           1       0.90      0.92      0.91       804
           2       0.82      0.78      0.80       196
           3       0.88      0.90      0.89       324
           4       0.86      0.84      0.85       290
           5       0.82      0.76      0.79        86

    accuracy           

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag of Words
bow = CountVectorizer(max_features=10000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print("BoW Shapes:")
print(X_train_bow.shape)
print(X_test_bow.shape)

BoW Shapes:
(13600, 10000)
(2400, 10000)


In [21]:
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

nb_bow_pred = nb_bow.predict(X_test_bow)

print("=== BoW Naive Bayes ===")
print(f"Accuracy: {accuracy_score(y_test, nb_bow_pred):.4f}")

=== BoW Naive Bayes ===
Accuracy: 0.8042


In [22]:
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)

lr_bow_pred = lr_bow.predict(X_test_bow)

print("=== BoW Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, lr_bow_pred):.4f}")

=== BoW Logistic Regression ===
Accuracy: 0.8888


In [23]:
svm_bow = LinearSVC()
svm_bow.fit(X_train_bow, y_train)

svm_bow_pred = svm_bow.predict(X_test_bow)

print("=== BoW SVM ===")
print(f"Accuracy: {accuracy_score(y_test, svm_bow_pred):.4f}")

=== BoW SVM ===
Accuracy: 0.8883


In [24]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.9 MB/s eta 0:00:00


In [25]:
from gensim.models import Word2Vec
import numpy as np

In [26]:
tokenized_text = [text.split() for text in df["clean_text"]]

print(tokenized_text[:2])

[['didnt', 'feel', 'humiliated'], ['go', 'feeling', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake']]


In [27]:
[['didnt', 'feel', 'humiliated'],
 ['go', 'feeling', 'hopeless', ...]]

[['didnt', 'feel', 'humiliated'], ['go', 'feeling', 'hopeless', Ellipsis]]

In [28]:
w2v_model = Word2Vec(
    sentences=tokenized_text,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print("Word2Vec trained successfully!")

Word2Vec trained successfully!


In [29]:
def document_vector(doc):
    words = doc.split()

    word_vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if len(word_vectors) == 0:
        return np.zeros(100)

    return np.mean(word_vectors, axis=0)

X_w2v = np.array([
    document_vector(text)
    for text in df["clean_text"]
])

print(X_w2v.shape)

(16000, 100)


In [30]:
from sklearn.model_selection import train_test_split

X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v,
    df["label"],
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

print(X_train_w2v.shape)
print(X_test_w2v.shape)

(13600, 100)
(2400, 100)


In [31]:
lr_w2v = LogisticRegression(max_iter=1000)

lr_w2v.fit(X_train_w2v, y_train_w2v)

lr_w2v_pred = lr_w2v.predict(X_test_w2v)

print("=== Word2Vec + Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test_w2v, lr_w2v_pred):.4f}")

=== Word2Vec + Logistic Regression ===
Accuracy: 0.3521


In [32]:
svm_w2v = LinearSVC()

svm_w2v.fit(X_train_w2v, y_train_w2v)

svm_w2v_pred = svm_w2v.predict(X_test_w2v)

print("=== Word2Vec + SVM ===")
print(f"Accuracy: {accuracy_score(y_test_w2v, svm_w2v_pred):.4f}")

=== Word2Vec + SVM ===
Accuracy: 0.3837


In [33]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-06-11 13:32:56--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-11 13:32:56--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  2.45MB/s    in 2m 56s  

2026-06-11 13:35:53 (4.68 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflatin

In [34]:
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = coefs

print("Loaded %s word vectors." % len(embeddings_index))

Loaded 400000 word vectors.


In [35]:
def glove_document_vector(doc):
    words = doc.split()

    vectors = [
        embeddings_index[word]
        for word in words
        if word in embeddings_index
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X_glove = np.array([
    glove_document_vector(text)
    for text in df["clean_text"]
])

print(X_glove.shape)

(16000, 100)


In [36]:
X_train_glove, X_test_glove, y_train_glove, y_test_glove = train_test_split(
    X_glove,
    df["label"],
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

print(X_train_glove.shape)
print(X_test_glove.shape)

(13600, 100)
(2400, 100)


In [37]:
lr_glove = LogisticRegression(max_iter=1000)

lr_glove.fit(X_train_glove, y_train_glove)

lr_glove_pred = lr_glove.predict(X_test_glove)

print("=== GloVe + Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test_glove, lr_glove_pred):.4f}")

=== GloVe + Logistic Regression ===
Accuracy: 0.5904


In [38]:
svm_glove = LinearSVC()

svm_glove.fit(X_train_glove, y_train_glove)

svm_glove_pred = svm_glove.predict(X_test_glove)

print("=== GloVe + SVM ===")
print(f"Accuracy: {accuracy_score(y_test_glove, svm_glove_pred):.4f}")

=== GloVe + SVM ===
Accuracy: 0.5746


In [39]:
from sklearn.ensemble import RandomForestClassifier

rf_glove = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_glove.fit(X_train_glove, y_train_glove)

rf_glove_pred = rf_glove.predict(X_test_glove)

print("=== GloVe + Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test_glove, rf_glove_pred):.4f}")

=== GloVe + Random Forest ===
Accuracy: 0.5217


In [40]:
import pandas as pd

results = pd.DataFrame({
    "Method": [
        "TF-IDF + Naive Bayes",
        "TF-IDF + Logistic Regression",
        "TF-IDF + SVM",
        "BoW + Naive Bayes",
        "BoW + Logistic Regression",
        "BoW + SVM",
        "Word2Vec + Logistic Regression",
        "Word2Vec + SVM",
        "GloVe + Logistic Regression",
        "GloVe + SVM",
        "GloVe + Random Forest"
    ],
    "Accuracy": [
        0.6871,
        0.8712,
        0.8946,
        0.8042,
        0.8888,
        0.8883,
        0.3521,
        0.3837,
        0.5904,
        0.5746,
        0.5217
    ]
})

results.sort_values("Accuracy", ascending=False)

,Method,Accuracy
2,TF-IDF + SVM,0.8946
4,BoW + Logistic Regression,0.8888
5,BoW + SVM,0.8883
1,TF-IDF + Logistic Regression,0.8712
3,BoW + Naive Bayes,0.8042
0,TF-IDF + Naive Bayes,0.6871
8,GloVe + Logistic Regression,0.5904
9,GloVe + SVM,0.5746
10,GloVe + Random Forest,0.5217
7,Word2Vec + SVM,0.3837
